# 阶段 8：学生模型还剩多少语言能力

学生只在"调度提示词 → 一个字母"上训练过。这个 notebook 比较**没训练的基座**和**合并了 LoRA 的学生**，看训练有没有损害原来的语言能力。全部数据内置在 notebook 里，不用上传任何东西。

**直接「代码执行程序 → 全部运行」**。运行时选带 GPU 的（G4、A100、L4 最好；T4 也能跑，会自动改用 float32）。

量四样东西，每个模型都量：

| 项目 | 做法 | 看什么 |
|---|---|---|
| 调度自检 | 8 条调度提示词，取第一个 token 里概率最大的合法字母 | 学生应该基本全对。它不对，说明 LoRA 没载对，后面的数都不能用 |
| 普通文本的困惑度 | 中文、英文各 8 段内置短文，不加对话格式 | 学生的困惑度比基座高多少 |
| 对话 | 20 个普通问题，贪心生成最多 48 个 token；分别用 Qwen 默认系统提示词和调度器的系统提示词 | 回答是正常句子，还是只有一个字母 |
| 选择题 | 32 道内置常识题（中英各 16，四个选项正确答案各占 1/4） | 两种读法：看 A/B/C/D 哪个字母概率最大；不用字母，看哪个选项的原文概率最大 |

第 7 格还会试着下载两份公开数据（wikitext-2 和 MMLU 的 200 道题）多量一遍。下载不了会自动跳过，不影响内置部分。

预计 8 个模型一共约 30 分钟（估算，没实测过）。

**结果**在 Drive 的 `MyDrive/COMLLM/s8_runs/language_check/`：每个模型一个 `model_<键>.json`（汇总）和 `items_<键>.json`（逐条回答），最后一格写 `report.md`。断线了重新「全部运行」会接着跑，跑完的模型自动跳过。

**止损线**：
- 任何一格报错就停，截图发到对话里。
- 某个**学生**的调度自检低于 6/8：停下，先查 LoRA 是不是载对了。
- 跑完「代码执行程序 → 断开连接并删除运行时」。


In [ ]:
# 第 1 格：安装固定版本的库（和阶段 6 的 Colab 闭环同样的版本）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.20.2", "peft==0.21.2", "accelerate", "sentencepiece"], check=True)
    # Colab 预装的 torchao 版本太老，peft 0.21 检测到它会直接报错。这里用不到 torchao，卸掉
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao 还在：菜单「代码执行程序 → 重新启动会话」后再「全部运行」"
print("依赖安装完成")


In [ ]:
# 第 2 格：检查 GPU，定精度（支持 bf16 就用 bf16，和闭环一样；T4 不支持，改用 float32）
import os, time, json, glob, math, random
from datetime import datetime, timezone
import torch

assert torch.cuda.is_available(), "没检测到 GPU：菜单「代码执行程序 → 更改运行时类型」选一个带 GPU 的"
DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name()
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
print("GPU:", GPU_NAME, "| torch", torch.__version__, "| 精度", str(DTYPE).replace("torch.", ""))


In [ ]:
# 第 3 格：跑哪些模型。一般不用改
RUN = "all"           # all = 2 个基座 + 6 个学生；也可以写逗号分隔的模型键，如 "base_0.5b,main_0.5b"
PUBLIC_DATA = True    # True = 第 7 格试着下载 wikitext-2 和 MMLU 多量一遍；下载不了自动跳过

# ---- 下面不用改 ----
DRIVE = "/content/drive/MyDrive/COMLLM"
OUT = f"{DRIVE}/s8_runs/language_check"
MAX_NEW_TOKENS = 48
WINDOW = 512          # 困惑度一次最多看多少个 token
BASES = {   # 基座: (Hugging Face 上的模型, 钉死的提交号，和训练时一样)
    "0.5B": ("Qwen/Qwen2.5-0.5B-Instruct", "7ae557604adf67be50417f59c2c2f167def9a775"),
    "1.5B": ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
}
MODELS = {  # 模型键: 基座、LoRA 运行目录的前缀（基座没有）、训练 notebook 存在 config.json 里的名字
    "base_0.5b":    {"base": "0.5B"},
    "base_1.5b":    {"base": "1.5B"},
    "main_0.5b":    {"base": "0.5B", "prefix": "main_0.5b_",    "name": "0.5b_soft_T0.3"},
    "soft_1.5b":    {"base": "1.5B", "prefix": "soft_1.5b_",    "name": "1.5b_soft_T0.3"},
    "hard_1.5b":    {"base": "1.5B", "prefix": "hard_1.5b_",    "name": "1.5b_hard"},
    "ps_0.5b":      {"base": "0.5B", "prefix": "ps_0.5b_",      "name": "0.5b_soft_T0.3_ps"},
    "ps_soft_1.5b": {"base": "1.5B", "prefix": "ps_soft_1.5b_", "name": "1.5b_soft_T0.3_ps"},
    "ps_hard_1.5b": {"base": "1.5B", "prefix": "ps_hard_1.5b_", "name": "1.5b_hard_ps"},
}
KEYS = list(MODELS) if RUN == "all" else [k.strip() for k in RUN.split(",")]
assert all(k in MODELS for k in KEYS), "RUN 里有不认识的模型键"
print("要跑：", KEYS)


In [ ]:
# 第 4 格：内置的测试数据和纯 Python 的小函数（不用 GPU；仓库里的单元测试会直接执行这一格）
import math

LETTERS = "ABCDEFGHIJKL"          # 调度动作的字母：A = 本地，B = 服务器 1，……
OPTION_LETTERS = "ABCD"           # 选择题的选项字母
QWEN_SYSTEM = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
SCHEDULER_SYSTEM = (
    "你是边缘计算任务卸载调度器。时间单位是槽（0.1秒），截止期10槽。"
    "A=本地执行（2.5GHz）；B、C、D…=卸载到服务器1、2、3…，要先上传（1.4Mbits/槽）。"
    "每比特297个CPU周期；服务器由活跃任务平分算力，新任务加入后活跃数加1。"
    "选总延迟最小且不超过截止期的动作，只输出一个字母。"
)

# 调度自检：8 条测试状态（来自阶段 3 的测试集，最优动作比第二名至少快 1 槽），best 是最优动作的编号
SCHED = [
    {
        "user": "任务4.26Mbits 上传排队2.6\nA 本地 排队0.0\nB 41.8GHz 活跃2 积压6.3\nC 30.0GHz 活跃1 积压4.0\nD 22.0GHz 活跃1 积压0.8",
        "best": 0,
        "n_actions": 4,
        "state": {
            "d": 4.2647,
            "local_wait": 0.0,
            "net_wait": 2.604,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 2,
                    "backlog": 6.3136,
                    "history": [
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 1,
                    "backlog": 3.9633,
                    "history": [
                        1.0,
                        0.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0,
                        0.0,
                        1.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 1,
                    "backlog": 0.801,
                    "history": [
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务4.69Mbits 上传排队0.0\nA 本地 排队0.6\nB 41.8GHz 活跃2 积压7.9\nC 30.0GHz 活跃0 积压0.0\nD 22.0GHz 活跃1 积压4.8",
        "best": 2,
        "n_actions": 4,
        "state": {
            "d": 4.694,
            "local_wait": 0.5972,
            "net_wait": 0.0,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 2,
                    "backlog": 7.9235,
                    "history": [
                        2.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        2.0,
                        1.0,
                        0.0,
                        2.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 1,
                    "backlog": 4.7885,
                    "history": [
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务2.92Mbits 上传排队2.1\nA 本地 排队0.0\nB 41.8GHz 活跃1 积压3.2\nC 30.0GHz 活跃1 积压3.8\nD 36.0GHz 活跃1 积压5.0\nE 28.0GHz 活跃1 积压4.3\nF 22.0GHz 活跃0 积压0.0",
        "best": 0,
        "n_actions": 6,
        "state": {
            "d": 2.9167,
            "local_wait": 0.0,
            "net_wait": 2.0611,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 1,
                    "backlog": 3.1865,
                    "history": [
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 1,
                    "backlog": 3.7762,
                    "history": [
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 36.0,
                    "active": 1,
                    "backlog": 4.9662,
                    "history": [
                        1.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0
                    ]
                },
                {
                    "ghz": 28.0,
                    "active": 1,
                    "backlog": 4.2855,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务2.66Mbits 上传排队0.0\nA 本地 排队2.5\nB 41.8GHz 活跃2 积压7.5\nC 45.0GHz 活跃3 积压12.9\nD 30.0GHz 活跃2 积压8.2\nE 36.0GHz 活跃2 积压8.7\nF 28.0GHz 活跃1 积压0.0\nG 25.0GHz 活跃2 积压5.3\nH 22.0GHz 活跃2 积压6.8",
        "best": 5,
        "n_actions": 8,
        "state": {
            "d": 2.6639,
            "local_wait": 2.5111,
            "net_wait": 0.0,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 2,
                    "backlog": 7.4812,
                    "history": [
                        2.0,
                        2.0,
                        2.0,
                        2.0,
                        1.0,
                        2.0,
                        2.0,
                        3.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 45.0,
                    "active": 3,
                    "backlog": 12.9344,
                    "history": [
                        2.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        3.0,
                        2.0,
                        2.0,
                        3.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 2,
                    "backlog": 8.2051,
                    "history": [
                        1.0,
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 36.0,
                    "active": 2,
                    "backlog": 8.6687,
                    "history": [
                        2.0,
                        2.0,
                        3.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 28.0,
                    "active": 1,
                    "backlog": 0.034,
                    "history": [
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        2.0,
                        1.0
                    ]
                },
                {
                    "ghz": 25.0,
                    "active": 2,
                    "backlog": 5.3477,
                    "history": [
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 2,
                    "backlog": 6.8233,
                    "history": [
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务3.62Mbits 上传排队2.5\nA 本地 排队0.0\nB 41.8GHz 活跃1 积压2.2\nC 45.0GHz 活跃1 积压4.4\nD 30.0GHz 活跃1 积压4.8\nE 36.0GHz 活跃0 积压0.0\nF 28.0GHz 活跃1 积压4.2\nG 25.0GHz 活跃1 积压3.6\nH 22.0GHz 活跃1 积压4.5",
        "best": 0,
        "n_actions": 8,
        "state": {
            "d": 3.621,
            "local_wait": 0.0,
            "net_wait": 2.4649,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 1,
                    "backlog": 2.1663,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 45.0,
                    "active": 1,
                    "backlog": 4.3964,
                    "history": [
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 1,
                    "backlog": 4.7572,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        0.0,
                        0.0,
                        1.0,
                        0.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 36.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0
                    ]
                },
                {
                    "ghz": 28.0,
                    "active": 1,
                    "backlog": 4.2306,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 25.0,
                    "active": 1,
                    "backlog": 3.5869,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 1,
                    "backlog": 4.4802,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务4.09Mbits 上传排队0.0\nA 本地 排队3.4\nB 41.8GHz 活跃1 积压2.2\nC 45.0GHz 活跃2 积压8.3\nD 48.0GHz 活跃3 积压12.3\nE 30.0GHz 活跃2 积压2.8\nF 33.0GHz 活跃2 积压8.8\nG 36.0GHz 活跃2 积压4.4\nH 28.0GHz 活跃2 积压4.9\nI 25.0GHz 活跃2 积压5.6\nJ 22.0GHz 活跃1 积压3.2",
        "best": 1,
        "n_actions": 10,
        "state": {
            "d": 4.0922,
            "local_wait": 3.3628,
            "net_wait": 0.0,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 1,
                    "backlog": 2.2469,
                    "history": [
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 45.0,
                    "active": 2,
                    "backlog": 8.3196,
                    "history": [
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        1.0,
                        3.0,
                        1.0,
                        2.0,
                        1.0,
                        2.0
                    ]
                },
                {
                    "ghz": 48.0,
                    "active": 3,
                    "backlog": 12.2558,
                    "history": [
                        3.0,
                        2.0,
                        1.0,
                        1.0,
                        3.0,
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        3.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 2,
                    "backlog": 2.7827,
                    "history": [
                        1.0,
                        2.0,
                        1.0,
                        2.0,
                        1.0,
                        2.0,
                        2.0,
                        2.0,
                        1.0,
                        2.0
                    ]
                },
                {
                    "ghz": 33.0,
                    "active": 2,
                    "backlog": 8.7817,
                    "history": [
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0
                    ]
                },
                {
                    "ghz": 36.0,
                    "active": 2,
                    "backlog": 4.4089,
                    "history": [
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        2.0,
                        2.0,
                        0.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 28.0,
                    "active": 2,
                    "backlog": 4.9419,
                    "history": [
                        2.0,
                        2.0,
                        2.0,
                        0.0,
                        2.0,
                        2.0,
                        2.0,
                        1.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 25.0,
                    "active": 2,
                    "backlog": 5.6386,
                    "history": [
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        2.0,
                        1.0,
                        2.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 1,
                    "backlog": 3.2499,
                    "history": [
                        0.0,
                        2.0,
                        2.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        2.0,
                        2.0,
                        1.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务2.02Mbits 上传排队2.4\nA 本地 排队0.0\nB 41.8GHz 活跃1 积压3.0\nC 45.0GHz 活跃1 积压2.9\nD 48.0GHz 活跃1 积压3.8\nE 30.0GHz 活跃0 积压0.0\nF 33.0GHz 活跃0 积压0.0\nG 36.0GHz 活跃0 积压0.0\nH 39.0GHz 活跃1 积压3.1\nI 28.0GHz 活跃0 积压0.0\nJ 25.0GHz 活跃0 积压0.0\nK 22.0GHz 活跃0 积压0.0\nL 20.0GHz 活跃0 积压0.0",
        "best": 0,
        "n_actions": 12,
        "state": {
            "d": 2.016,
            "local_wait": 0.0,
            "net_wait": 2.3529,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 1,
                    "backlog": 3.0102,
                    "history": [
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 45.0,
                    "active": 1,
                    "backlog": 2.94,
                    "history": [
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 48.0,
                    "active": 1,
                    "backlog": 3.8086,
                    "history": [
                        0.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                },
                {
                    "ghz": 33.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                },
                {
                    "ghz": 36.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                },
                {
                    "ghz": 39.0,
                    "active": 1,
                    "backlog": 3.0658,
                    "history": [
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        1.0
                    ]
                },
                {
                    "ghz": 28.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                },
                {
                    "ghz": 25.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                },
                {
                    "ghz": 20.0,
                    "active": 0,
                    "backlog": 0,
                    "history": [
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ]
                }
            ]
        }
    },
    {
        "user": "任务3.87Mbits 上传排队0.0\nA 本地 排队4.1\nB 41.8GHz 活跃3 积压12.4\nC 45.0GHz 活跃3 积压11.1\nD 48.0GHz 活跃4 积压13.6\nE 30.0GHz 活跃2 积压8.5\nF 33.0GHz 活跃3 积压12.5\nG 36.0GHz 活跃4 积压13.3\nH 39.0GHz 活跃2 积压7.2\nI 28.0GHz 活跃3 积压10.7\nJ 25.0GHz 活跃2 积压7.0\nK 22.0GHz 活跃2 积压7.2\nL 20.0GHz 活跃3 积压12.8",
        "best": 7,
        "n_actions": 12,
        "state": {
            "d": 3.8699,
            "local_wait": 4.0513,
            "net_wait": 0.0,
            "rate": 1.4,
            "deadline": 10,
            "local_ghz": 2.5,
            "servers": [
                {
                    "ghz": 41.8,
                    "active": 3,
                    "backlog": 12.423,
                    "history": [
                        3.0,
                        2.0,
                        3.0,
                        4.0,
                        2.0,
                        3.0,
                        3.0,
                        3.0,
                        4.0,
                        3.0
                    ]
                },
                {
                    "ghz": 45.0,
                    "active": 3,
                    "backlog": 11.1457,
                    "history": [
                        3.0,
                        4.0,
                        3.0,
                        4.0,
                        3.0,
                        3.0,
                        3.0,
                        3.0,
                        2.0,
                        3.0
                    ]
                },
                {
                    "ghz": 48.0,
                    "active": 4,
                    "backlog": 13.633,
                    "history": [
                        2.0,
                        4.0,
                        3.0,
                        4.0,
                        3.0,
                        2.0,
                        4.0,
                        2.0,
                        3.0,
                        4.0
                    ]
                },
                {
                    "ghz": 30.0,
                    "active": 2,
                    "backlog": 8.5018,
                    "history": [
                        4.0,
                        3.0,
                        2.0,
                        1.0,
                        4.0,
                        4.0,
                        3.0,
                        1.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 33.0,
                    "active": 3,
                    "backlog": 12.5187,
                    "history": [
                        3.0,
                        3.0,
                        3.0,
                        1.0,
                        4.0,
                        4.0,
                        1.0,
                        2.0,
                        3.0,
                        3.0
                    ]
                },
                {
                    "ghz": 36.0,
                    "active": 4,
                    "backlog": 13.2815,
                    "history": [
                        4.0,
                        2.0,
                        2.0,
                        3.0,
                        4.0,
                        1.0,
                        4.0,
                        2.0,
                        2.0,
                        4.0
                    ]
                },
                {
                    "ghz": 39.0,
                    "active": 2,
                    "backlog": 7.2122,
                    "history": [
                        2.0,
                        4.0,
                        0.0,
                        4.0,
                        3.0,
                        4.0,
                        3.0,
                        2.0,
                        4.0,
                        2.0
                    ]
                },
                {
                    "ghz": 28.0,
                    "active": 3,
                    "backlog": 10.7009,
                    "history": [
                        2.0,
                        3.0,
                        3.0,
                        2.0,
                        3.0,
                        2.0,
                        3.0,
                        1.0,
                        3.0,
                        3.0
                    ]
                },
                {
                    "ghz": 25.0,
                    "active": 2,
                    "backlog": 7.0377,
                    "history": [
                        1.0,
                        3.0,
                        2.0,
                        3.0,
                        2.0,
                        3.0,
                        3.0,
                        2.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 22.0,
                    "active": 2,
                    "backlog": 7.2337,
                    "history": [
                        1.0,
                        2.0,
                        3.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0,
                        2.0
                    ]
                },
                {
                    "ghz": 20.0,
                    "active": 3,
                    "backlog": 12.7776,
                    "history": [
                        3.0,
                        3.0,
                        1.0,
                        2.0,
                        2.0,
                        2.0,
                        3.0,
                        3.0,
                        1.0,
                        3.0
                    ]
                }
            ]
        }
    }
]

# 普通短文：中文、英文各 8 段
PASSAGES = {
    "zh": [
        "清晨的公园里，许多人在跑步和打太极拳。树上的鸟叫个不停，草地上还带着露水。一位老人坐在长椅上看报纸，身边的小狗安静地趴着。",
        "做一碗番茄鸡蛋面并不难。先把鸡蛋炒熟盛出，再把番茄炒出汁，加水烧开后下面条，最后把鸡蛋倒回锅里，加一点盐就可以出锅了。",
        "图书馆是城市里最安静的地方之一。人们在这里借书、查资料，也有人只是找个座位自习。管理员会把归还的书重新放回书架。",
        "地球绕太阳转一圈大约需要三百六十五天，这就是一年。地球自己转一圈是一天，所以我们会看到白天和黑夜交替出现。",
        "坐火车旅行的好处是可以看窗外的风景。列车经过田野、河流和小镇，旅客可以看书、聊天，也可以靠着窗户睡一会儿。",
        "学习一门新语言需要耐心。每天记几个单词，多听多说，遇到不懂的地方及时查词典，时间长了自然会有进步。",
        "秋天到了，树叶慢慢变黄，一片片落在地上。农民忙着收割庄稼，果园里的苹果和梨也成熟了，到处是丰收的景象。",
        "计算机由硬件和软件组成。硬件包括处理器、内存和硬盘，软件则是运行在硬件上的程序，两者配合才能完成各种任务。",
    ],
    "en": [
        "The village market opens early every Saturday. Farmers bring vegetables, eggs and fresh bread, and neighbours stop to talk while they shop. By noon most of the stalls are empty and the square is quiet again.",
        "To make a cup of tea, boil some water and pour it over the tea leaves. Wait a few minutes, then remove the leaves. Some people add milk or sugar, while others prefer it plain.",
        "A bicycle is a simple machine with two wheels, a frame and a chain. When the rider pushes the pedals, the chain turns the back wheel and the bicycle moves forward.",
        "The moon travels around the Earth about once every month. It does not make its own light. We see it because it reflects light from the sun.",
        "Reading before bed can help people relax. A good story takes the mind away from the worries of the day, and many readers fall asleep more easily after a few pages.",
        "Rivers begin in the mountains as small streams. As they flow downhill, other streams join them, and they grow wider and slower before they finally reach the sea.",
        "A library card lets you borrow books for a few weeks at a time. If you return them late, you may have to pay a small fine, so it is wise to note the due date.",
        "Computers store information as numbers. Programs tell the processor which steps to follow, and the processor carries them out one after another at very high speed.",
    ],
}

# 普通对话问题：中文、英文各 10 个
CHAT = {
    "zh": [
        "请用一句话介绍一下你自己。", "写一句描写春天的话。", "把“今天天气很好”翻译成英文。", "3 加 5 等于几？请直接回答。",
        "请列举三种常见的水果。", "为什么天空是蓝色的？请用两句话解释。", "请写一句鼓励朋友的话。", "“高兴”的近义词是什么？",
        "请用“因为……所以……”造一个句子。", "一周有几天？",
    ],
    "en": [
        "Introduce yourself in one sentence.", "Write one sentence about the ocean.", "Translate \"good morning\" into Chinese.",
        "What is 9 minus 4? Answer directly.", "Name three colors.", "Why do we need sleep? Answer in two sentences.",
        "Write a short greeting for a new colleague.", "What is a synonym for \"happy\"?",
        "Complete the sentence: \"The cat sat on the ...\"", "How many hours are there in a day?",
    ],
}

# 常识选择题：(题目, 四个选项, 正确选项的下标)。中英各 16 道，正确答案在 A/B/C/D 上各 4 道
MCQ = {
    "zh": [
        ("中国的首都是哪座城市？", ["上海", "北京", "广州", "深圳"], 1),
        ("一年有多少个月？", ["10 个", "11 个", "12 个", "13 个"], 2),
        ("太阳从哪个方向升起？", ["东方", "西方", "南方", "北方"], 0),
        ("7 加 8 等于多少？", ["13", "14", "16", "15"], 3),
        ("下列哪一种是水果？", ["白菜", "土豆", "苹果", "萝卜"], 2),
        ("《西游记》的作者是谁？", ["吴承恩", "曹雪芹", "罗贯中", "施耐庵"], 0),
        ("水的化学式是什么？", ["CO2", "H2O", "O2", "NaCl"], 1),
        ("地球上最高的山峰是哪一座？", ["泰山", "富士山", "乞力马扎罗山", "珠穆朗玛峰"], 3),
        ("“大”的反义词是什么？", ["高", "长", "多", "小"], 3),
        ("一个小时有多少分钟？", ["60 分钟", "30 分钟", "100 分钟", "24 分钟"], 0),
        ("下列哪种动物会飞？", ["兔子", "麻雀", "乌龟", "金鱼"], 1),
        ("光在真空中的速度大约是每秒多少公里？", ["3 千公里", "3 万公里", "30 万公里", "300 万公里"], 2),
        ("中国的国庆节是几月几日？", ["1 月 1 日", "5 月 1 日", "10 月 1 日", "12 月 25 日"], 2),
        ("人体正常体温大约是多少摄氏度？", ["37 度", "27 度", "47 度", "57 度"], 0),
        ("100 除以 4 等于多少？", ["20", "25", "40", "50"], 1),
        ("长江最终流入哪片海域？", ["渤海", "黄海", "南海", "东海"], 3),
    ],
    "en": [
        ("What is the capital of France?", ["Paris", "Madrid", "Rome", "Berlin"], 0),
        ("How many days are there in a week?", ["five", "six", "seven", "eight"], 2),
        ("Which planet is known as the Red Planet?", ["Venus", "Mars", "Jupiter", "Saturn"], 1),
        ("What is 12 times 12?", ["124", "132", "140", "144"], 3),
        ("At what temperature in degrees Celsius does water freeze at standard pressure?", ["0", "32", "100", "50"], 0),
        ("Which of these animals is a mammal?", ["shark", "frog", "dolphin", "eagle"], 2),
        ("Who wrote the play \"Romeo and Juliet\"?", ["Charles Dickens", "William Shakespeare", "Mark Twain", "Jane Austen"], 1),
        ("What is the chemical symbol for gold?", ["Ag", "Fe", "Go", "Au"], 3),
        ("Which is the largest ocean on Earth?", ["the Atlantic Ocean", "the Indian Ocean", "the Arctic Ocean", "the Pacific Ocean"], 3),
        ("What is the opposite of \"hot\"?", ["cold", "warm", "dry", "bright"], 0),
        ("How many sides does a triangle have?", ["two", "three", "four", "five"], 1),
        ("Which gas do plants absorb from the air for photosynthesis?", ["oxygen", "nitrogen", "carbon dioxide", "helium"], 2),
        ("What is the past tense of the verb \"go\"?", ["goed", "gone", "went", "going"], 2),
        ("Which month comes right after March?", ["April", "May", "February", "June"], 0),
        ("What is the square root of 81?", ["7", "8", "10", "9"], 3),
        ("On which continent is Egypt located?", ["Asia", "Africa", "Europe", "South America"], 1),
    ],
}


def chatml(messages):
    """消息列表 → 一整段提示词，末尾停在 assistant 开头。和仓库 deploy/prompts.py 的 chatml 一样，训练和部署都是这个拼法。"""
    text = ""
    for m in messages:
        text = text + "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    return text + "<|im_start|>assistant\n"


def chat_prompt(system, user):
    return chatml([{"role": "system", "content": system}, {"role": "user", "content": user}])


def mcq_letter_prompt(lang, question, options):
    """选择题的对话提示词：题目 + 四行选项 + "只回答字母"。"""
    lines = [question]
    for i, opt in enumerate(options):
        lines.append(f"{OPTION_LETTERS[i]}. {opt}")
    lines.append("请只回答选项字母。" if lang == "zh" else "Answer with the option letter only.")
    return chat_prompt(QWEN_SYSTEM, "\n".join(lines))


def mcq_cloze(lang, question, option):
    """不用字母的读法：纯文本"问题……答案：" + 选项原文。返回 (前文, 要打分的续写)。"""
    if lang == "zh":
        return f"问题：{question}\n答案：", option
    return f"Question: {question}\nAnswer:", " " + option


def classify_response(text):
    """一段回答归成三类：empty = 空；letter = 去掉空白和句末标点后只剩一个调度字母（A–L）；text = 其他。"""
    t = text.strip().rstrip("。.!！").strip()
    if not t:
        return "empty"
    if len(t) == 1 and t.upper() in LETTERS:
        return "letter"
    return "text"


def chat_stats(items):
    """items：[{"response": 文本, "letter_mass": 第一个 token 落在调度字母上的概率}, ...] → 汇总。"""
    n = len(items)
    kinds = [classify_response(x["response"]) for x in items]
    return {"n": n, "letter": kinds.count("letter"), "empty": kinds.count("empty"), "text": kinds.count("text"),
            "letter_mass": sum(x["letter_mass"] for x in items) / n,
            "mean_chars": sum(len(x["response"].strip()) for x in items) / n}


def accuracy(preds, answers):
    assert len(preds) == len(answers) and answers
    return sum(1 for p, a in zip(preds, answers) if p == a) / len(answers)


def argmax(xs):
    best = 0
    for i, x in enumerate(xs):
        if x > xs[best]:
            best = i
    return best


def perplexity(logprobs):
    """一串 token 的对数概率 → 困惑度。"""
    return math.exp(-sum(logprobs) / len(logprobs))


def base_of(key):
    return "base_0.5b" if "0.5b" in key else "base_1.5b"


def fmt(x, nd=1):
    return "–" if x is None else f"{x:.{nd}f}"


def report_markdown(results):
    """results：{模型键: 第 8 格存的汇总字典} → Markdown 报告。学生那几行的困惑度后面括号里是"÷ 同尺寸基座"。"""
    order = [k for k in ("base_0.5b", "main_0.5b", "ps_0.5b", "base_1.5b", "soft_1.5b", "hard_1.5b", "ps_soft_1.5b",
                         "ps_hard_1.5b") if k in results]
    lines = ["# 语言能力检查：基座 vs 学生", "",
             "由 `train/S8_语言能力检查.ipynb` 生成。困惑度越低越好；括号里是学生 ÷ 同尺寸基座。", "",
             "| 模型 | 调度自检 | 困惑度 中文 | 困惑度 英文 | 对话只答一个字母（默认系统提示 / 调度系统提示） | "
             "第一个 token 落在调度字母上的概率（默认 / 调度） | 选择题·看字母 中 / 英 | 选择题·看选项原文 中 / 英 |",
             "|---|---|---|---|---|---|---|---|"]
    for k in order:
        r = results[k]
        b = results.get(base_of(k))
        cells = [k, f"{round(r['sched']['acc'] * r['sched']['n'])}/{r['sched']['n']}"]
        for lang in ("zh", "en"):
            p = r["ppl"][lang]["ppl"]
            ratio = "" if b is None or b is r else f"（×{p / b['ppl'][lang]['ppl']:.2f}）"
            cells.append(f"{p:.1f}{ratio}")
        d, s = r["chat"]["default"], r["chat"]["scheduler"]
        cells.append(f"{d['letter']}/{d['n']} · {s['letter']}/{s['n']}")
        cells.append(f"{100 * d['letter_mass']:.1f}% · {100 * s['letter_mass']:.1f}%")
        cells.append(f"{100 * r['mcq']['zh']['letter_acc']:.0f}% / {100 * r['mcq']['en']['letter_acc']:.0f}%")
        cells.append(f"{100 * r['mcq']['zh']['cloze_acc']:.0f}% / {100 * r['mcq']['en']['cloze_acc']:.0f}%")
        lines.append("| " + " | ".join(cells) + " |")
    pub = [k for k in order if results[k]["ppl"].get("wikitext") or results[k]["mcq"].get("mmlu")]
    if pub:
        lines += ["", "## 公开数据（下载成功才有）", "",
                  "| 模型 | wikitext-2 困惑度 | MMLU 200 题·看字母 | MMLU 200 题·看选项原文 |", "|---|---|---|---|"]
        for k in pub:
            r = results[k]
            w, m = r["ppl"].get("wikitext"), r["mcq"].get("mmlu")
            lines.append(f"| {k} | {fmt(w['ppl'] if w else None)} | {fmt(100 * m['letter_acc'] if m else None)}% | "
                         f"{fmt(100 * m['cloze_acc'] if m else None)}% |")
    lines += ["", "选择题四个选项的正确答案各占 1/4，瞎猜或者永远答同一个字母是 25%。",
              "内置数据量小（短文各 8 段、对话 20 问、选择题 32 道），只能看出大的变化。"]
    return "\n".join(lines) + "\n"


print(f"内置数据：调度自检 {len(SCHED)} 条，短文 {len(PASSAGES['zh'])} + {len(PASSAGES['en'])} 段，"
      f"对话 {len(CHAT['zh'])} + {len(CHAT['en'])} 问，选择题 {len(MCQ['zh'])} + {len(MCQ['en'])} 道")


In [ ]:
# 第 5 格：挂 Drive，找每个学生的 LoRA（取前缀相同的运行目录里最新的一个，名字要对得上）
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)


def find_adapter(key):
    m = MODELS[key]
    dirs = sorted(d for d in glob.glob(os.path.join(f"{DRIVE}/s3_runs", m["prefix"] + "*"))
                  if os.path.isdir(os.path.join(d, "adapter")))
    assert dirs, f"{DRIVE}/s3_runs 下没有 {m['prefix']}* 的运行目录（要有 adapter/）"
    run = dirs[-1]
    with open(os.path.join(run, "config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    assert cfg.get("name") == m["name"], f"{run} 的 config 名字是 {cfg.get('name')}，应该是 {m['name']}"
    return run


for key in KEYS:
    print(f"{key:14s} →", find_adapter(key) if MODELS[key].get("prefix") else "基座，不加 LoRA")


In [ ]:
# 第 6 格：载入模型和打分用的函数
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import transformers, peft


def load_model(key):
    """返回 (分词器, 模型, LoRA 运行目录)。学生 = 基座合并 LoRA，和 Colab 闭环的做法一样；基座的运行目录是空字符串。"""
    m = MODELS[key]
    base_path, rev = BASES[m["base"]]
    tok = AutoTokenizer.from_pretrained(base_path, revision=rev)
    model = AutoModelForCausalLM.from_pretrained(base_path, revision=rev, dtype=DTYPE).to(DEVICE)
    run = ""
    if m.get("prefix"):
        run = find_adapter(key)
        model = PeftModel.from_pretrained(model, os.path.join(run, "adapter")).merge_and_unload()
    model.eval()
    return tok, model, run


def encode(tok, text):
    return tok(text, add_special_tokens=False)["input_ids"]


def single_token_ids(tok, letters):
    ids = []
    for c in letters:
        x = encode(tok, c)
        assert len(x) == 1, f"{c} 不是单 token：{x}"
        ids.append(x[0])
    return ids


@torch.no_grad()
def next_logprobs(model, ids):
    """一串 token → 每个位置上"下一个 token"的对数概率，形状 (长度, 词表)。"""
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    return torch.log_softmax(model(input_ids=x).logits[0].float(), dim=-1)


def cont_logprobs(model, prefix_ids, cont_ids):
    """在 prefix 后面接 cont，返回 cont 里每个 token 的对数概率（列表）。"""
    assert prefix_ids and cont_ids
    lp = next_logprobs(model, prefix_ids + cont_ids)
    out = []
    for i, t in enumerate(cont_ids):
        out.append(lp[len(prefix_ids) - 1 + i, t].item())
    return out


def text_logprobs(model, tok, texts):
    """几段纯文本每个 token 的对数概率（每段的第一个 token 没有上文，不算）；长文本按 WINDOW 切开。"""
    out = []
    for text in texts:
        ids = encode(tok, text)
        for start in range(0, len(ids), WINDOW):
            w = ids[start:start + WINDOW]
            if len(w) >= 2:
                out += cont_logprobs(model, w[:1], w[1:])
    return out


def first_token_logprobs(model, tok, prompt):
    return next_logprobs(model, encode(tok, prompt))[-1]


@torch.no_grad()
def generate(model, tok, prompt, im_end):
    """贪心生成，遇到 <|im_end|> 停。重复惩罚设成 1.0，不用模型自带的默认值。"""
    ids = encode(tok, prompt)
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    pad = tok.pad_token_id if tok.pad_token_id is not None else im_end
    out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), do_sample=False,
                         max_new_tokens=MAX_NEW_TOKENS, eos_token_id=im_end, pad_token_id=pad, repetition_penalty=1.0)
    return tok.decode(out[0, len(ids):], skip_special_tokens=True)


def score_mcq(model, tok, lang, items, option_ids):
    """一组选择题 → (逐题记录, 看字母的准确率, 看选项原文的准确率)。

    看字母：对话提示词后第一个 token，在 A/B/C/D 四个字母里取概率最大的
    看原文：纯文本"问题……答案："后面接每个选项的原文，取平均每 token 对数概率最大的
    """
    rows, letter_preds, cloze_preds, answers = [], [], [], []
    for question, options, answer in items:
        lp = first_token_logprobs(model, tok, mcq_letter_prompt(lang, question, options))
        letter_scores = [lp[i].item() for i in option_ids]
        cloze_scores = []
        for opt in options:
            prefix, cont = mcq_cloze(lang, question, opt)
            lps = cont_logprobs(model, encode(tok, prefix), encode(tok, cont))
            cloze_scores.append(sum(lps) / len(lps))
        letter_preds.append(argmax(letter_scores))
        cloze_preds.append(argmax(cloze_scores))
        answers.append(answer)
        rows.append({"question": question, "answer": answer, "letter_pred": letter_preds[-1],
                     "cloze_pred": cloze_preds[-1],
                     "abcd_mass": sum(math.exp(s) for s in letter_scores)})
    return rows, accuracy(letter_preds, answers), accuracy(cloze_preds, answers)


print("transformers", transformers.__version__, "| peft", peft.__version__)


In [ ]:
# 第 7 格：公开数据（可选）。下载不了就跳过，不影响内置部分
PUBLIC = {"wikitext": None, "mmlu": None}
if PUBLIC_DATA:
    try:
        from datasets import load_dataset
        wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
        text = "\n".join(t for t in wt["text"] if t.strip())
        PUBLIC["wikitext"] = text[:80000]          # 约 2 万个 token
        print("wikitext-2 测试集：取前", len(PUBLIC["wikitext"]), "个字符")
    except Exception as e:      # noqa: BLE001
        print("wikitext-2 下载失败，跳过：", repr(e)[:200])
    try:
        from datasets import load_dataset
        mm = load_dataset("cais/mmlu", "all", split="test")
        idx = list(range(len(mm)))
        random.Random(0).shuffle(idx)
        PUBLIC["mmlu"] = []
        for i in idx[:200]:
            row = mm[i]
            assert len(row["choices"]) == 4 and 0 <= int(row["answer"]) < 4
            PUBLIC["mmlu"].append((row["question"], list(row["choices"]), int(row["answer"])))
        print("MMLU 测试集：随机取", len(PUBLIC["mmlu"]), "道（种子 0）")
    except Exception as e:      # noqa: BLE001
        PUBLIC["mmlu"] = None
        print("MMLU 下载失败，跳过：", repr(e)[:200])


In [ ]:
# 第 8 格：逐个模型量（跑完一个存一个，断了重跑会跳过已经存好的）
for key in KEYS:
    path = f"{OUT}/model_{key}.json"
    if os.path.exists(path):
        print(key, "已经跑过，跳过")
        continue
    t0 = time.time()
    print(f"\n===== {key} =====", flush=True)
    tok, model, run = load_model(key)
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    action_ids = single_token_ids(tok, LETTERS)
    option_ids = single_token_ids(tok, OPTION_LETTERS)
    items = {"sched": [], "chat": {"default": [], "scheduler": []}, "mcq": {}}

    # 1. 调度自检：合法字母里概率最大的那个是不是最优动作
    hits = 0
    for ex in SCHED:
        lp = first_token_logprobs(model, tok, chat_prompt(SCHEDULER_SYSTEM, ex["user"]))
        legal = [lp[i].item() for i in action_ids[:ex["n_actions"]]]
        pred = argmax(legal)
        hits += int(pred == ex["best"])
        items["sched"].append({"best": ex["best"], "pred": pred,
                               "letter_mass": sum(math.exp(lp[i].item()) for i in action_ids)})
    sched = {"n": len(SCHED), "acc": hits / len(SCHED),
             "letter_mass": sum(x["letter_mass"] for x in items["sched"]) / len(SCHED)}
    print(f"调度自检 {hits}/{len(SCHED)}", flush=True)

    # 2. 普通文本的困惑度
    ppl = {}
    for lang in ("zh", "en"):
        lps = text_logprobs(model, tok, PASSAGES[lang])
        ppl[lang] = {"ppl": perplexity(lps), "tokens": len(lps)}
    ppl["wikitext"] = None
    if PUBLIC["wikitext"]:
        lps = text_logprobs(model, tok, [PUBLIC["wikitext"]])
        ppl["wikitext"] = {"ppl": perplexity(lps), "tokens": len(lps)}
    print("困惑度 中文 %.1f  英文 %.1f" % (ppl["zh"]["ppl"], ppl["en"]["ppl"]), flush=True)

    # 3. 对话：两种系统提示词各问一遍
    chat = {}
    for cond, system in (("default", QWEN_SYSTEM), ("scheduler", SCHEDULER_SYSTEM)):
        for lang in ("zh", "en"):
            for q in CHAT[lang]:
                prompt = chat_prompt(system, q)
                lp = first_token_logprobs(model, tok, prompt)
                items["chat"][cond].append({"lang": lang, "question": q, "response": generate(model, tok, prompt, im_end),
                                            "letter_mass": sum(math.exp(lp[i].item()) for i in action_ids)})
        chat[cond] = chat_stats(items["chat"][cond])
    print("对话只答一个字母：默认系统提示 %d/%d，调度系统提示 %d/%d" % (
        chat["default"]["letter"], chat["default"]["n"], chat["scheduler"]["letter"], chat["scheduler"]["n"]), flush=True)

    # 4. 选择题
    mcq = {"mmlu": None}
    for lang in ("zh", "en"):
        rows, a_letter, a_cloze = score_mcq(model, tok, lang, MCQ[lang], option_ids)
        items["mcq"][lang] = rows
        mcq[lang] = {"n": len(rows), "letter_acc": a_letter, "cloze_acc": a_cloze}
    if PUBLIC["mmlu"]:
        rows, a_letter, a_cloze = score_mcq(model, tok, "en", PUBLIC["mmlu"], option_ids)
        items["mcq"]["mmlu"] = [{k: v for k, v in r.items() if k != "question"} for r in rows]
        mcq["mmlu"] = {"n": len(rows), "letter_acc": a_letter, "cloze_acc": a_cloze}
    print("选择题 看字母 中 %.0f%% 英 %.0f%%；看选项原文 中 %.0f%% 英 %.0f%%" % (
        100 * mcq["zh"]["letter_acc"], 100 * mcq["en"]["letter_acc"],
        100 * mcq["zh"]["cloze_acc"], 100 * mcq["en"]["cloze_acc"]), flush=True)

    result = {"key": key, "base": list(BASES[MODELS[key]["base"]]), "adapter_run": run,
              "dtype": str(DTYPE).replace("torch.", ""), "gpu": GPU_NAME, "torch": torch.__version__,
              "transformers": transformers.__version__, "peft": peft.__version__,
              "max_new_tokens": MAX_NEW_TOKENS, "window": WINDOW,
              "finished_utc": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
              "seconds": round(time.time() - t0, 1), "sched": sched, "ppl": ppl, "chat": chat, "mcq": mcq}
    with open(f"{OUT}/items_{key}.json", "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=1)
    with open(path, "w", encoding="utf-8") as f:       # 汇总最后写：有它就算这个模型跑完了
        json.dump(result, f, ensure_ascii=False, indent=1)
    del model
    torch.cuda.empty_cache()
    print(f"{key} 用时 {(time.time() - t0) / 60:.1f} 分钟", flush=True)


In [ ]:
# 第 9 格：汇总成一张表，写进 report.md
results = {}
for p in sorted(glob.glob(f"{OUT}/model_*.json")):
    with open(p, encoding="utf-8") as f:
        r = json.load(f)
    results[r["key"]] = r
report = report_markdown(results)
with open(f"{OUT}/report.md", "w", encoding="utf-8") as f:
    f.write(report)
print(report)
for k, r in results.items():
    if MODELS[k].get("prefix") and r["sched"]["acc"] < 0.75:
        print(f"注意：{k} 的调度自检只有 {r['sched']['acc']:.0%}，先查 LoRA 是不是载对了")


## 跑完之后

1. 把第 9 格的表截图，或者直接说"跑完了"。结果都在 Drive 的 `MyDrive/COMLLM/s8_runs/language_check/`，可以从那里取。
2. 菜单「代码执行程序 → 断开连接并删除运行时」，停止扣额度。

读数提醒：
- 困惑度是在不加对话格式的纯文本上量的；对话和选择题·看字母用的是训练时那种对话格式，所以这两类可能给出不同的结论。
- 选择题·看字母走的是学生被训练过的"第一个 token 输出字母"这条通道，学生可能把它当成调度题来答；看选项原文不走这条通道。
- 内置数据量小，只能看出大的变化；公开数据那两项下载成功的话更可靠一些。
